In [1]:
conda install pandas matplotlib seaborn numpy

Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 23.9.0
  latest version: 24.3.0

Please update conda by running

    $ conda update -n base -c conda-forge conda

Or to minimize the number of packages updated during conda update use

     conda install conda=24.3.0



# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [10]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
# Reading In and Checking Data
eggnog_COG_category_df = pd.read_csv('out.emapper.annotations_modified', header=4, sep='\t', usecols=['#query', 'COG_category'])

eggnog_COG_category_df_non_null = eggnog_COG_category_df.dropna(subset=['COG_category','#query'])

# Filter rows where 'COG_category' is exactly one uppercase letter
cog_letters = eggnog_COG_category_df_non_null['COG_category'].apply(lambda x: len(x) == 1 and x.isupper())

# The result is a boolean series, so we use it to filter eggnog_COG_category_df_non_null
df_single_letter_COG_category = eggnog_COG_category_df_non_null[cog_letters]
print("\nSingle Letter COG Category Data Frame\n",df_single_letter_COG_category)

print("\nEggnog COG Category Data Frame\n",eggnog_COG_category_df.head())

# Read in tsv file
context_df = pd.read_csv('Coarse_OrthoGroup_Contexts.Without_Last_Column.tsv', sep='\t', usecols=['Context entropy score', 'OG'])

print("\TSV file Data Frame\n",context_df.head())

# Merge filtered data frame with corresponding ortholog groups 
merged_df = pd.merge(df_single_letter_COG_category, context_df, left_on='#query', right_on='OG', how='inner')

print("\Merged Data Frame\n",merged_df.head())

# Specifically identify the orthologous genes with the highest and lowest context entropy scores, 
# and find their corresponding descriptions
top_10_genes_context_entropy = merged_df.nlargest(10, 'Context entropy score')
bottom_10_genes_context_entropy = merged_df.nsmallest(10, 'Context entropy score')
print("\nTOP 10\n", top_10_genes_context_entropy)
print("\nBOTTOM 10\n", bottom_10_genes_context_entropy)

# Read the entire dataset into a pandas DataFrame
description_df = pd.read_csv('Coarse_OrthoGroup_Contexts_COG_Description_Added.tsv', sep='\t', header=0)

# Strip whitespace from the column names
description_df.columns = [col.strip() for col in description_df.columns]

# Index the DataFrame to only keep the 'OG' and 'description' columns
og_description_df = description_df[['OG', 'description']]

# Now og_description_df is a DataFrame with just 'OG' and 'description' columns
print("\OG Descriptions\n",og_description_df.head())

# Merge the top 10 genes DataFrame with the descriptions DataFrame
merged_top_10 = pd.merge(top_10_genes_context_entropy, og_description_df, left_on='#query', right_on='OG', how='inner')

# Merge the bottom 10 genes DataFrame with the descriptions DataFrame
merged_bottom_10 = pd.merge(bottom_10_genes_context_entropy, og_description_df, left_on='#query', right_on='OG', how='inner')

print("\nMERGED BOTTOM 10\n", merged_bottom_10)
print("\nMERGED TOP 10\n", merged_top_10, "\n")

         #query COG_category
0     OG0002476            Q
1     OG0003244            S
3     OG0000817            H
4     OG0004555            C
5     OG0001370            D
...         ...          ...
5747  OG0007066            I
5748  OG0007068            E
5749  OG0007327            V
5750  OG0007350            H
5751  OG0005570            K

[4610 rows x 2 columns]
      #query COG_category
0  OG0002476            Q
1  OG0003244            S
2  OG0003030            -
3  OG0000817            H
4  OG0004555            C
          OG  Context entropy score
0  OG0000000                   6.95
1  OG0000001                   5.53
2  OG0000002                   6.84
3  OG0000003                   5.53
4  OG0000004                   6.46
      #query COG_category         OG  Context entropy score
0  OG0002476            Q  OG0002476                   4.67
1  OG0003244            S  OG0003244                   2.85
2  OG0000817            H  OG0000817                   3.46
3  OG0004555   